# Missing Values and Duplicates Cleaning

In [31]:
#import libraries
import pandas as pd
import numpy as np
import re
import emoji 
import string
from lingua import Language, LanguageDetectorBuilder

#import nlp libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

#download nltk resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
df = pd.read_csv("MASTER_RAW_kenya_fintech.csv")

# Preview dataset
df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,NaN,mpesa


## Checking Missing Values

This step identifies columns with null or missing values that may affect preprocessing and model training.

In [32]:
missing_values = df.isnull().sum()

missing_values

reviewId                    0
userName                    0
userImage                   0
content                     1
score                       0
thumbsUpCount               0
reviewCreatedVersion     6164
at                          0
replyContent            29574
repliedAt               29574
appVersion               6164
app_name                    0
dtype: int64

## Removing Empty Reviews

Rows with completely empty review texts are removed because they do not contribute meaningful information for sentiment analysis.

In [33]:
df = df[df['content'].notna()]

df.shape

(53506, 12)

## Handling Missing App Versions

Missing app versions are replaced with 'Unknown' to preserve dataset consistency.

In [34]:
df['appVersion'] = df['appVersion'].fillna('Unknown')

df['appVersion'].isnull().sum()

0

## Checking Duplicate Review IDs

Each review should have a unique review ID. Duplicate IDs may indicate repeated records.

In [35]:
duplicate_ids = df.duplicated(subset=['reviewId']).sum()

print("Duplicate Review IDs:", duplicate_ids)

Duplicate Review IDs: 0


## Checking Duplicate Users and Posts

This step checks whether the same users repeatedly posted identical reviews.

In [36]:
duplicate_users_posts = df.duplicated(
    subset=['userName', 'content']
).sum()

print("Duplicate Users/Posts:", duplicate_users_posts)

Duplicate Users/Posts: 18055


## Removing Unnecessary Columns

Columns such as usernames and user images are removed because they are not required for sentiment classification.

In [37]:
df = df.drop(columns=['userImage', 'userName'])

df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa


## Final Dataset Validation

A final check is performed to confirm the dataset is clean and ready for preprocessing and modelling.

In [38]:
df.isnull().sum()

reviewId                    0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion     6164
at                          0
replyContent            29573
repliedAt               29573
appVersion                  0
app_name                    0
dtype: int64

In [39]:
cleaned_df=pd.read_csv("cleaned_kenya_fintech.csv")
cleaned_df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name,cleaned_text,final_language,normalized_text
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa,the app still has issues on otp because i have...,english,the app still has issues on otp because i have...
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa,si everytime nitakuwa na bundles za ku check m...,english,si everytime nitakuwa na bundles za ku check m...
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa,this is the stupidest app ever from saf the worst,english,this is the stupidest app ever from saf the worst
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa,life must go on without this useless app it us...,english,life must go on without this useless app it us...
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa,the upgrade is terrible,english,the upgrade is terrible


## Implementing text normalization function

In [40]:
def clean_text(text):
    text = str(text)  
    
    
    text = text.lower() # converting text to lowercase
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    
    
    text = re.sub(r'\d+', '', text)  #Removing numbers
    
    
    text = text.translate(str.maketrans('', '', string.punctuation))# Remove punctuations
    
    
    text = emoji.replace_emoji(text, replace='')# Remove emojis
    
    # Removing  special characters 
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()# remove spaces
    
    # Removing extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

## Applying the cleaning function

In [41]:
df["cleaned_text"] = df["content"].apply(clean_text)

df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name,cleaned_text
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa,the app still has issues on otp because i have...
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa,si everytime nitakuwa na bundles za ku check m...
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa,this is the stupidest app ever from saf the worst
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa,life must go on without this useless app it us...
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa,the upgrade is terrible


The text data was preprocessed by converting it to lowercase, removing URLs, emojis, and non-alphabetic characters, and eliminating extra spaces to ensure consistency. The cleaned data is now suitable for further analysis and machine learning.

### Language detection

This function implements a hierarchical detection strategy that prioritizes local context by flagging specific Kenyan keywords and "Sheng" terminology before defaulting to probabilistic modeling. By layering a custom heuristic over the Lingua detector, the pipeline successfully isolates informal code-switching and financial slang that standard libraries often misclassify. This refined categorization provides a more granular sociolinguistic breakdown, distinguishing formal English and Swahili from the colloquialisms prevalent in Kenyan digital discourse.

In [42]:
# Defining the languages expected in your Kenyan dataset
languages = [Language.ENGLISH, Language.SWAHILI]

# 2. Build the detector
# We use 'with_low_accuracy_mode_disabled' for better results on short texts
detector = LanguageDetectorBuilder.from_languages(*languages).build()

def detect_with_lingua(text):
    text = str(text)
    if not text.strip():
        return "unknown"
    
    # Get the most likely language
    confidence_values = detector.compute_language_confidence_values(text)
    
    # Lingua returns a list of results; we take the top one
    if not confidence_values:
        return "unknown"
        
    top_lang = confidence_values[0].language.name.lower()
    
    # Lingua returns 'swahili' or 'english'
    if top_lang == 'swahili':
        return 'sw'
    if top_lang == 'english':
        return 'en'
    
    return "unknown"

In [43]:
def final_kenyan_detector(text):
    text = str(text) # Already lowercased per your previous message
    words = set(text.split())

    # Light Sheng normalisation — covers the most common terms
    sheng_words = ['pesa', 'hela', 'fedha', 
                   'dough', 'mulla', 'mbaya', 'poa', 
                   'safi', 'best', 'nzuri', 'waizi', 'wezi', 
                   'wameiba', 'wameniibia', 'imenishinda', 'imenichukua', 
                   'umeniibia', 'nirudishie', 'yangu', 'app imekufa', 'haifanyi', 
                   'inashindwa', 'imenikatia', 'nipewe', 'niambie', 'sana', 
                   'kabisa', 'deni', 'mkopo', 'riba']
    
    # 1. Check for Sheng/Mixed Keywords first (High Priority)
    sheng_hits = len(words.intersection(sheng_words))
    #sw_hits = len(words.intersection(sw))
    
    # 2. Use Lingua for the base language
    base_lang = detect_with_lingua(text)
    
    # --- LOGIC ---
    
    # If it has Sheng keywords, it's Sheng (or Mixed)
    if sheng_hits > 0:
        return "sheng"
    
    # If Lingua says Swahili OR we hit Swahili keywords
    if base_lang == "sw": #or sw_hits > 0:
        return "swahili"
    
    # If Lingua says English
    if base_lang == "en":
        return "english"
    
    return "unknown"

# Apply to your dataframe
df["final_language"] = df["cleaned_text"].apply(final_kenyan_detector)
print(df["final_language"].value_counts())

final_language
english    46590
sheng       4948
swahili     1308
unknown      660
Name: count, dtype: int64


A significant portion of the modules relies on localized financial slang and code-switching, which standard NLP tools typically overlook. Consequently, the refined distribution provides a more accurate sociolinguistic profile of the Kenyan dataset, highlighting a dominant English presence alongside a robust and distinct informal dialect.

#### Sheng normalization

In [44]:
SHENG_MAP = {
    # Money related
    'pesa':     'money',
    'hela':     'money',
    'fedha':    'money',
    'dough':    'money',
    'mulla':    'money',

    # Bad / good
    'mbaya':    'bad',
    'poa':      'good',
    'safi':     'good',
    'best':     'best',
    'nzuri':     'good',

    # People / actions
    'waizi':    'thieves',
    'wezi':     'thieves',
    'wameiba':  'stolen',
    'wameniibia': 'cheated me',
    'imenishinda': 'frustrated',
    'imenichukua': 'taken from me',
    'umeniibia': 'you cheated me',
    'nirudishie': 'refund me',
    'yangu':     'mine',

    # App related
    'app imekufa': 'app is dead',
    'haifanyi':  'not working',
    'inashindwa': 'failing',
    'imenikatia': 'declined me',
    'nipewe':    'give me',
    'niambie':   'tell me',
    'sana':     'very',
    'kabisa':    'much',

    # Loan related
    'deni':     'debt',
    'mkopo':    'loan',
    'riba':     'interest',
}
def apply_sheng_map(text):
    """
    Replaces Sheng and Swahili slang terms with English equivalents.
    Must be called AFTER lowercasing and BEFORE removing special characters.
    """
    if not isinstance(text, str):
        return text

    # Sort by length descending so longer phrases match before shorter words
    # e.g. 'mbaya sana' matches before 'mbaya'
    sorted_map = sorted(SHENG_MAP.items(),
                        key=lambda x: len(x[0]),
                        reverse=True)

    for sheng, english in sorted_map:
        text = re.sub(rf'\b{re.escape(sheng)}\b', english, text)

    return text
def apply_sheng_map(text):
    """
    Replaces Sheng and Swahili slang terms with English equivalents.
    Must be called AFTER lowercasing and BEFORE removing special characters.
    """
    if not isinstance(text, str):
        return text

    # Sort by length descending so longer phrases match before shorter words
    # e.g. 'mbaya sana' matches before 'mbaya'
    sorted_map = sorted(SHENG_MAP.items(),
                        key=lambda x: len(x[0]),
                        reverse=True)

    for sheng, english in sorted_map:
        text = re.sub(rf'\b{re.escape(sheng)}\b', english, text)

    return text
import re

# Applying the mapping to create a normalized column
df['normalized_text'] = df['cleaned_text'].apply(apply_sheng_map)

#### Natural Language Preprocessing
In this section we use a function to prepare raw customer reviews for machine learning and NLP analysis by transforming unstructured text into a cleaner and more meaningful format.
The preprocessing pipeline performs the following tasks:
- Tokenization -splits review text into individual words called tokens.
- Stopword Removal-Removes very common words that add little meaning
- Removes very short words that usually have little analytical value
- Stemming - Reduces words to their root form
- Lemmatization-Converts words into their proper dictionary form
- Create processed_text Column-combines the cleaned tokens back into a single processed sentence

In [45]:
# stopwords

stop_words = set(stopwords.words('english'))

# Custom fintech stopwords

custom_stopwords = {
    'app',
    'please',
    'safaricom',
    'mpesa'
}

stop_words.update(custom_stopwords)

# Stemmer

stemmer = PorterStemmer()

# Lemmatizer

lemmatizer = WordNetLemmatizer()


#Create the preprocessing function

def preprocess_text(text):
    
    # Convert to string
    
    text = str(text)
    

#Tokenization
    
    tokens = word_tokenize(text)
    
    # Stopword Removal
    tokens_no_stopwords = [
    word
    for word in tokens
    if word.lower() not in stop_words
    and word.isalpha()
]
    
    # Remove short meaningless tokens
    meaningful_tokens = [
        word
        for word in tokens_no_stopwords
        if len(word) > 2
    ]
    
    # Stemming
    
    stemmed_tokens = [
        stemmer.stem(word)
        for word in meaningful_tokens
    ]
    
    # Lemmatization
    
    lemmatized_tokens = [
        lemmatizer.lemmatize(word)
        for word in meaningful_tokens
    ]
    
    # Create a final processed text version (lemmatized)
    
    processed_text = ' '.join(lemmatized_tokens)
    
    # Return all outputs
    
    return pd.Series([
        tokens,
        tokens_no_stopwords,
        meaningful_tokens,
        stemmed_tokens,
        lemmatized_tokens,
        processed_text
    ])

# Apply the preprocessing function to the 'content' column and create new columns for each output
df[[
    'tokens',
    'tokens_no_stopwords',
    'meaningful_tokens',
    'stemmed_tokens',
    'lemmatized_tokens',
    'processed_text'
]] = df['content'].apply(preprocess_text)


#Preview the new columns
df[[
    'content',
    'tokens',
    'tokens_no_stopwords',
    'meaningful_tokens',
    'stemmed_tokens',
    'lemmatized_tokens',
    'processed_text']].head(10)

,content,tokens,tokens_no_stopwords,meaningful_tokens,stemmed_tokens,lemmatized_tokens,processed_text
0,"The app still has issues on OTP, because I hav...","[The, app, still, has, issues, on, OTP, ,, bec...","[still, issues, OTP, received, OTP, login, tri...","[still, issues, OTP, received, OTP, login, tri...","[still, issu, otp, receiv, otp, login, tri, ev...","[still, issue, OTP, received, OTP, login, trie...",still issue OTP received OTP login tried every...
1,si everytime nitakuwa na bundles za ku check m...,"[si, everytime, nitakuwa, na, bundles, za, ku,...","[si, everytime, nitakuwa, na, bundles, za, ku,...","[everytime, nitakuwa, bundles, check, number, ...","[everytim, nitakuwa, bundl, check, number, tri...","[everytime, nitakuwa, bundle, check, number, t...",everytime nitakuwa bundle check number try fix...
2,this is the stupidest app ever from saf. the w...,"[this, is, the, stupidest, app, ever, from, sa...","[stupidest, ever, saf, worst]","[stupidest, ever, saf, worst]","[stupidest, ever, saf, worst]","[stupidest, ever, saf, worst]",stupidest ever saf worst
3,Life must go on without this useless app. It u...,"[Life, must, go, on, without, this, useless, a...","[Life, must, go, without, useless, used, work,...","[Life, must, without, useless, used, work, wel...","[life, must, without, useless, use, work, well...","[Life, must, without, useless, used, work, wel...",Life must without useless used work well data ...
4,the upgrade is terrible,"[the, upgrade, is, terrible]","[upgrade, terrible]","[upgrade, terrible]","[upgrad, terribl]","[upgrade, terrible]",upgrade terrible
5,"Worst app I have used logs me out Everytime, I...","[Worst, app, I, have, used, logs, me, out, Eve...","[Worst, used, logs, Everytime, use, number, su...","[Worst, used, logs, Everytime, use, number, su...","[worst, use, log, everytim, use, number, sum, ...","[Worst, used, log, Everytime, use, number, sum...",Worst used log Everytime use number sum one ye...
6,terrible! if u still have the old version plac...,"[terrible, !, if, u, still, have, the, old, ve...","[terrible, u, still, old, version, place, update]","[terrible, still, old, version, place, update]","[terribl, still, old, version, place, updat]","[terrible, still, old, version, place, update]",terrible still old version place update
7,Poor service. Can you please work on making th...,"[Poor, service, ., Can, you, please, work, on,...","[Poor, service, work, making, offline, functio...","[Poor, service, work, making, offline, functio...","[poor, servic, work, make, offlin, function, a...","[Poor, service, work, making, offline, functio...",Poor service work making offline functionality...
8,best,[best],[best],[best],[best],[best],best
9,Terrible user experience. Bring back the old M...,"[Terrible, user, experience, ., Bring, back, t...","[Terrible, user, experience, Bring, back, old]","[Terrible, user, experience, Bring, back, old]","[terribl, user, experi, bring, back, old]","[Terrible, user, experience, Bring, back, old]",Terrible user experience Bring back old


The preprocessing pipeline successfully transformed raw fintech reviews into clean, structured text suitable for NLP modeling. Tokenization split the text into individual words, while stopword removal and elimination of short meaningless words reduced noise and improved data quality.

Punctuation and unnecessary symbols were effectively removed, resulting in cleaner tokens for analysis. Stemming produced root word forms (e.g., terrible → terribl), while lemmatization preserved more readable and meaningful word forms (e.g., issues → issue).

The processed_text column provides a normalized representation of each review and is ready for feature extraction methods such as TF-IDF and machine learning models like Logistic Regression, XGBoost, and topic modelling.

### Feature Engineering

#### 1.Sentiment Labels from Score Ratings 
This is your target variable — not a feature. Every model you train predicts this. Without it you have no supervised learning problem.

Used by: Logistic Regression, XGBoost, AfriBERTa

In [46]:
# Create sentiment labels based on the 'score' column
def sentiment_label(score):
    if score >= 4: return 'positive'
    if score == 3: return 'average'
    return 'negative'

df['sentiment_label'] = df['score'].apply(sentiment_label)

print(df['sentiment_label'].value_counts())

sentiment_label
positive    40296
negative    11040
average      2170
Name: count, dtype: int64


#### 2.Complaint Labels 
This is your secondary target variable. It gives your project a second classification task and directly feeds your business recommendations.

Used by: XGBoost

In [47]:
def complaint_label(text):
    if not isinstance(text, str): return 'general'
    text = text.lower()
    if any(w in text for w in ['fraud','scam','stolen','wameiba','waizi']):
        return 'fraud'
    if any(w in text for w in ['fail','error','stuck','decline','haifanyi']):
        return 'service_failure'
    if any(w in text for w in ['loan','fuliza','mkopo','riba','interest']):
        return 'loan_complaint'
    if any(w in text for w in ['crash','hang','slow','update']):
        return 'app_issue'
    return 'general'

df['complaint_label'] = df['processed_text'].apply(complaint_label)

print(df['complaint_label'].value_counts())

complaint_label
general            46251
loan_complaint      4055
app_issue           2470
service_failure      594
fraud                136
Name: count, dtype: int64


#### 3.Binary fraud indicator
It checks whether a review mentions fraud or not

Used by: XGBoost features, Financial Distress Index, business recommendations


In [ ]:
# Fraud indicator
fraud_words = ['fraud', 'scam', 'waizi', 'stolen', 'wameiba', 'imeibiwa', 'nilibiwa', 'cheat', 'cheated', 'swindled']

df['fraud_indicator'] = df['processed_text'].str.contains(
    '|'.join(fraud_words), case=False, na=False
).astype(int)

print('Fraud indicator distribution:')
print(df['fraud_indicator'].value_counts())

Fraud indicator distribution:
fraud_indicator
0    53355
1      151
Name: count, dtype: int64
